## **AutoML com H2o**

In [1]:
import h2o
from h2o.automl import H2OAutoML
import pandas as pd

In [2]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.18" 2026-01-20; OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-1); OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-1, mixed mode, sharing)
  Starting server from /home/joao-inac-io/AutoML-com--Python/.venv/lib/python3.10/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpqmcvvfnx
  JVM stdout: /tmp/tmpqmcvvfnx/h2o_joao_inac_io_started_from_python.out
  JVM stderr: /tmp/tmpqmcvvfnx/h2o_joao_inac_io_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,04 secs
H2O_cluster_timezone:,America/Fortaleza
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 28 days
H2O_cluster_name:,H2O_from_python_joao_inac_io_0qhr54
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4.727 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [3]:
df = pd.read_csv("../data/Churn_treino.csv", sep=";")
df = h2o.H2OFrame(df)
df

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
619,France,Female,42,2,0,1,1,1,1.01349e+07,1
608,Spain,Female,41,1,8.38079e+06,1,0,1,1.12543e+07,0
502,France,Female,42,8,1.59661e+06,3,1,0,1.13932e+07,1
699,France,Female,39,1,0,2,0,0,9.38266e+06,0
850,Spain,Female,43,2,1.25511e+07,1,1,1,790841,0
645,Spain,Male,44,8,1.13756e+07,2,1,0,1.49757e+07,1
822,France,Male,50,7,0,2,1,1,100628,0
376,Germany,Female,29,4,1.15047e+07,4,1,0,1.19347e+07,1
501,France,Male,44,4,1.42051e+07,2,0,1,749405,0
684,France,Male,27,2,1.34604e+07,1,1,1,7.17257e+06,0


In [4]:
treino, teste = df.split_frame(ratios=[0.7])

In [5]:
treino["Exited"] = treino["Exited"].asfactor()
teste["Exited"] = teste["Exited"].asfactor()

In [6]:
modelo = H2OAutoML(max_runtime_secs=180)
modelo.train(y="Exited", training_frame=treino)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),7/12
# GBM base models (used / total),3/5
# XGBoost base models (used / total),2/3
# DeepLearning base models (used / total),1/1
# DRF base models (used / total),1/2
# GLM base models (used / total),0/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [7]:
ranking = modelo.leaderboard
ranking = ranking.as_data_frame(use_multi_thread=True)
ranking

Export File progress: |██████████████████████████████████████████████████████████| (done) 100%


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,StackedEnsemble_AllModels_2_AutoML_1_20260510_...,0.858844,0.340825,0.696948,0.229631,0.321958,0.103657
1,StackedEnsemble_AllModels_3_AutoML_1_20260510_...,0.858424,0.340787,0.698441,0.234474,0.321611,0.103434
2,StackedEnsemble_BestOfFamily_3_AutoML_1_202605...,0.858079,0.340985,0.696940,0.231784,0.321875,0.103603
3,GBM_5_AutoML_1_20260510_182836,0.856552,0.343510,0.692164,0.239359,0.322763,0.104176
4,StackedEnsemble_AllModels_1_AutoML_1_20260510_...,0.856196,0.343942,0.692022,0.239554,0.323593,0.104713
5,GBM_grid_1_AutoML_1_20260510_182836_model_5,0.856109,0.347084,0.687426,0.236772,0.325021,0.105639
6,StackedEnsemble_BestOfFamily_2_AutoML_1_202605...,0.855482,0.345406,0.690049,0.229714,0.324353,0.105205
7,StackedEnsemble_BestOfFamily_1_AutoML_1_202605...,0.855247,0.345677,0.689281,0.231406,0.324439,0.105261
8,GBM_grid_1_AutoML_1_20260510_182836_model_6,0.855183,0.347690,0.689832,0.241343,0.325102,0.105691
9,GBM_1_AutoML_1_20260510_182836,0.854820,0.346471,0.689439,0.243777,0.324772,0.105477


In [8]:
df_test = pd.read_csv("../data/Churn_teste.csv", sep=";")
df_test = h2o.H2OFrame(teste)

In [9]:
prever = modelo.leader.predict(df_test)
prever = prever.as_data_frame(use_multi_thread=True)
prever

stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
Export File progress: |██████████████████████████████████████████████████████████| (done) 100%


,predict,p0,p1
0,0,0.941189,0.058811
1,0,0.892592,0.107408
2,0,0.989695,0.010305
3,0,0.981925,0.018075
4,0,0.977666,0.022334
...,...,...,...
2989,0,0.724747,0.275253
2990,0,0.963648,0.036352
2991,0,0.856080,0.143920
2992,0,0.736438,0.263562
